# 03 — Feature Engineering for Next-Day AQI Forecasting

## Objective

This notebook transforms cleaned city-level AQI observations into features for next-day regression and classification.

The dataset covers **April 2022 to April 2025**. Features capture historical AQI behaviour, seasonal variation, and geographic and pollutant context.

Two prediction targets are constructed:

- **Regression:** Next-day AQI (`target_AQI_next`).
- **Classification:** Next-day air-quality category (`target_bucket_next`).

All predictors are restricted to information available on the prediction date to prevent future-data leakage.

**Input:** `data/Cleaned/aqi_2023_2025_clean.csv`  
**Output:** `data/Cleaned/aqi_2023_25_features.csv`

In [1]:
import pandas as pd
import numpy as np
import os, sys
sys.path.append("..")
from src.aqi_common import ALL_FEATURES, FEATURE_GROUPS

pd.set_option('display.max_columns', None)
IN_PATH = "../data/Cleaned/aqi_2023_2025_clean.csv"
OUT_PATH = "../data/Cleaned/aqi_2023_25_features.csv"


## Step 1 — Load and tidy the cleaned file
Works whether your cleaned file kept the raw Kaggle column names or already uses `City`, `AQI`, `AQI_Bucket`. It also tries both date formats.

In [2]:
raw = pd.read_csv(IN_PATH)
df = raw.rename(columns={'area': 'City', 'aqi_value': 'AQI', 'air_quality_status': 'AQI_Bucket', 'date': 'Date'})

d = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
if d.isna().mean() > 0.5:
    d = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df['Date'] = d
df['AQI'] = pd.to_numeric(df['AQI'], errors='coerce')

for col, default in [('state', 'Unknown'), ('number_of_monitoring_stations', np.nan), ('prominent_pollutants', np.nan)]:
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=['City', 'Date', 'AQI', 'AQI_Bucket'])
df['City'] = df['City'].astype(str).str.strip().str.title()
df['AQI_Bucket'] = df['AQI_Bucket'].astype(str).str.strip().str.title()
df = df.drop_duplicates(subset=['City', 'Date'])

df = df[['City', 'state', 'Date', 'AQI', 'AQI_Bucket', 'number_of_monitoring_stations', 'prominent_pollutants']]
df = df.sort_values(['City', 'Date']).reset_index(drop=True)
print("Rows:", len(df), "| cities:", df['City'].nunique(), "| dates:", df['Date'].min().date(), "to", df['Date'].max().date())
df.head()

Rows: 235005 | cities: 291 | dates: 2022-04-01 to 2025-04-30


,City,state,Date,AQI,AQI_Bucket,number_of_monitoring_stations,prominent_pollutants
0,Agartala,Tripura,2022-04-01,78,Satisfactory,2,"PM10,SO2"
1,Agartala,Tripura,2022-04-02,77,Satisfactory,2,"PM10,SO2"
2,Agartala,Tripura,2022-04-03,76,Satisfactory,2,"PM2.5,PM10"
3,Agartala,Tripura,2022-04-04,57,Satisfactory,2,PM10
4,Agartala,Tripura,2022-04-05,64,Satisfactory,2,PM10


## Step 2 — Temporal Continuity Assessment

Lagged features require consistent daily intervals. Initial inspection shows that **93.65% of consecutive city observations are one day apart**, indicating gaps in the remaining observations.

These gaps must be addressed before generating lag and rolling features to preserve their intended temporal meaning.

In [3]:
gap = df.groupby('City')['Date'].diff().dt.days.dropna()
print("Days between consecutive rows (should mostly be 1):")
print(gap.value_counts().sort_index().head(10))
print("\nShare of consecutive rows exactly 1 day apart:", round((gap == 1).mean() * 100, 2), "%")

Days between consecutive rows (should mostly be 1):
Date
1.0     219804
2.0       9241
3.0       2706
4.0       1143
5.0        582
6.0        336
7.0        210
8.0        138
9.0         97
10.0        75
Name: count, dtype: int64

Share of consecutive rows exactly 1 day apart: 93.65 %


## Step 3 — Daily Time-Series Alignment

Each city's observations are reindexed to a continuous daily calendar. This introduces **36,900 missing-date rows**, increasing the intermediate dataset from 235,005 to 271,905 records.

Missing AQI values are preserved rather than interpolated, preventing artificial observations from influencing historical features. Rows without the required information are excluded during the final feature-selection stage.

In [4]:
parts = []
for city, g in df.groupby('City', sort=False):
    idx = pd.date_range(g['Date'].min(), g['Date'].max(), freq='D')
    g = g.set_index('Date').reindex(idx)
    g.index.name = 'Date'
    g['City'] = city
    parts.append(g.reset_index())
full = pd.concat(parts, ignore_index=True)

print("Rows before:", len(df), "| after adding missing days:", len(full))
print("Empty rows added:", full['AQI'].isna().sum())

# static columns: carry the city's value into the empty rows
full['state'] = full.groupby('City')['state'].transform(lambda s: s.ffill().bfill())
st = full.groupby('City')['number_of_monitoring_stations'].transform(lambda s: s.ffill().bfill())
full['number_of_monitoring_stations'] = st.fillna(st.median())

Rows before: 235005 | after adding missing days: 271905
Empty rows added: 36900


## Step 4 — Calendar features
`is_festival` marks Diwali ±3 days. Your data covers Diwali 2023 and 2024 (2025's is listed for completeness but falls after the data ends).

In [5]:
def month_to_season(m):
    if m in [12, 1, 2]: return 'Winter'
    if m in [3, 4, 5]: return 'Summer'
    if m in [6, 7, 8, 9]: return 'Monsoon'
    return 'Post-Monsoon'

full['month'] = full['Date'].dt.month
full['season'] = full['month'].apply(month_to_season)
full['is_winter'] = full['month'].isin([11, 12, 1, 2]).astype(int)
full['weekday'] = full['Date'].dt.dayofweek
full['is_weekend'] = (full['weekday'] >= 5).astype(int)

diwali = pd.to_datetime(['2023-11-12', '2024-11-01', '2025-10-20'])
days_apart = np.abs((full['Date'].to_numpy()[:, None] - diwali.to_numpy()[None, :]) / np.timedelta64(1, 'D'))
full['is_festival'] = (days_apart.min(axis=1) <= 3).astype(int)

print("Festival rows:", full['is_festival'].sum())
full[['Date', 'month', 'season', 'is_winter', 'weekday', 'is_weekend', 'is_festival']].head()

Festival rows: 3824


,Date,month,season,is_winter,weekday,is_weekend,is_festival
0,2022-04-01,4,Summer,0,4,0,0
1,2022-04-02,4,Summer,0,5,1,0
2,2022-04-03,4,Summer,0,6,1,0
3,2022-04-04,4,Summer,0,0,0,0
4,2022-04-05,4,Summer,0,1,0,0


## Step 5 — Lag and rolling features
- `AQI` itself is **today's** value (known today, so allowed).
- `AQI_lagK` = AQI K days ago.
- Rolling statistics use `shift(1)` first, so the window ends **yesterday**.
- `min_periods` lets a window with a few missing days still produce a value.

In [6]:
full = full.sort_values(['City', 'Date']).reset_index(drop=True)
g = full.groupby('City')['AQI']

for k in [1, 2, 3, 7]:
    full[f'AQI_lag{k}'] = g.shift(k)

for w in [3, 7, 14]:
    full[f'AQI_roll{w}'] = g.transform(lambda x, w=w: x.shift(1).rolling(w, min_periods=max(2, int(w * 0.7))).mean())
full['AQI_std7'] = g.transform(lambda x: x.shift(1).rolling(7, min_periods=5).std())
full['AQI_change1'] = full['AQI'] - full['AQI_lag1']

full[['City', 'Date', 'AQI', 'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7', 'AQI_roll3', 'AQI_roll7', 'AQI_roll14']].head(12)

,City,Date,AQI,AQI_lag1,AQI_lag2,AQI_lag3,AQI_lag7,AQI_roll3,AQI_roll7,AQI_roll14
0,Agartala,2022-04-01,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Agartala,2022-04-02,77.0,78.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Agartala,2022-04-03,76.0,77.0,78.0,NaN,NaN,77.500000,NaN,NaN
3,Agartala,2022-04-04,57.0,76.0,77.0,78.0,NaN,77.000000,NaN,NaN
4,Agartala,2022-04-05,64.0,57.0,76.0,77.0,NaN,70.000000,72.000000,NaN
5,Agartala,2022-04-06,58.0,64.0,57.0,76.0,NaN,65.666667,70.400000,NaN
6,Agartala,2022-04-07,73.0,58.0,64.0,57.0,NaN,59.666667,68.333333,NaN
7,Agartala,2022-04-08,98.0,73.0,58.0,64.0,78.0,65.000000,69.000000,NaN
8,Agartala,2022-04-09,82.0,98.0,73.0,58.0,77.0,76.333333,71.857143,NaN
9,Agartala,2022-04-10,48.0,82.0,98.0,73.0,76.0,84.333333,72.571429,73.666667


## Step 6 — Context features
The dominant pollutant is known today, so **today's** and **yesterday's** values are both allowed. We keep the six most common pollutants and group the rest as `Other`.

In [7]:
first = full['prominent_pollutants'].astype('string').str.split(',').str[0].str.strip()
top = first.value_counts().head(6).index
pol = first.where(first.isin(top), 'Other')
pol = pol.where(first.notna())              # empty (gap) rows stay empty
full['pollutant_today'] = pol
full['pollutant_prev'] = full.groupby('City')['pollutant_today'].shift(1)
full['pollutant_today'] = full['pollutant_today'].fillna('Unknown').astype(str)
full['pollutant_prev'] = full['pollutant_prev'].fillna('Unknown').astype(str)
full['state'] = full['state'].fillna('Unknown').astype(str)

print(full['pollutant_today'].value_counts())

pollutant_today
PM10       120227
PM2.5       76333
Unknown     36900
O3          17184
CO          13814
SO2          3923
NO2          3439
Other          85
Name: count, dtype: int64


## Step 7 — Targets
`shift(-1)` looks one day **forward**. Targets are the only place that is allowed to look ahead, and they are never used as features.

In [8]:
full['target_AQI_next'] = full.groupby('City')['AQI'].shift(-1)
full['target_bucket_next'] = full.groupby('City')['AQI_Bucket'].shift(-1)
full[['City', 'Date', 'AQI', 'AQI_Bucket', 'target_AQI_next', 'target_bucket_next']].head(5)

,City,Date,AQI,AQI_Bucket,target_AQI_next,target_bucket_next
0,Agartala,2022-04-01,78.0,Satisfactory,77.0,Satisfactory
1,Agartala,2022-04-02,77.0,Satisfactory,76.0,Satisfactory
2,Agartala,2022-04-03,76.0,Satisfactory,57.0,Satisfactory
3,Agartala,2022-04-04,57.0,Satisfactory,64.0,Satisfactory
4,Agartala,2022-04-05,64.0,Satisfactory,58.0,Satisfactory


## Step 8 — Selection of Valid Modelling Observations

Observations with missing required predictors or next-day targets are excluded to construct a complete modelling dataset.

After filtering, **173,464 observations** remain from the intermediate 271,905 city-day records. This ensures that the selected observations contain the historical information required by the modelling pipeline.

The resulting dataset retains the temporal and contextual features used in subsequent experiments.

In [9]:
need = ['AQI', 'AQI_lag1', 'AQI_lag2', 'AQI_lag3', 'AQI_lag7',
        'AQI_roll3', 'AQI_roll7', 'AQI_roll14', 'AQI_std7', 'AQI_change1',
        'target_AQI_next', 'target_bucket_next']

before = len(full)
feat = full.dropna(subset=need).copy()
print("Rows before:", before, "| usable rows:", len(feat), "| removed:", before - len(feat))

keep = ['City', 'Date'] + [c for c in ALL_FEATURES if c != 'City'] + ['AQI_Bucket', 'target_AQI_next', 'target_bucket_next']
feat = feat[keep].reset_index(drop=True)
feat.head()

Rows before: 271905 | usable rows: 173464 | removed: 98441


,City,Date,AQI,AQI_change1,AQI_lag1,AQI_lag2,AQI_lag3,AQI_lag7,AQI_roll3,AQI_roll7,AQI_roll14,AQI_std7,month,weekday,is_weekend,is_winter,is_festival,season,state,number_of_monitoring_stations,pollutant_today,pollutant_prev,AQI_Bucket,target_AQI_next,target_bucket_next
0,Agartala,2022-04-10,48.0,-34.0,82.0,98.0,73.0,76.0,84.333333,72.571429,73.666667,14.581462,4,6,1,0,0,Summer,Tripura,2.0,PM10,PM10,Good,54.0,Satisfactory
1,Agartala,2022-04-11,54.0,6.0,48.0,82.0,98.0,57.0,76.000000,68.571429,71.100000,17.106111,4,0,0,0,0,Summer,Tripura,2.0,PM10,PM10,Satisfactory,62.0,Satisfactory
2,Agartala,2022-04-12,62.0,8.0,54.0,48.0,82.0,64.0,61.333333,68.142857,69.545455,17.477877,4,1,0,0,0,Summer,Tripura,1.0,PM10,PM10,Satisfactory,65.0,Satisfactory
3,Agartala,2022-04-13,65.0,3.0,62.0,54.0,48.0,58.0,54.666667,67.857143,68.916667,17.572977,4,2,0,0,0,Summer,Tripura,2.0,PM10,PM10,Satisfactory,56.0,Satisfactory
4,Agartala,2022-04-14,56.0,-9.0,65.0,62.0,54.0,73.0,60.333333,68.857143,68.615385,17.111678,4,3,0,0,0,Summer,Tripura,2.0,PM10,PM10,Satisfactory,50.0,Good


## Step 9 — Trap 2: leakage tests
We rebuild three values by hand from the original data and check they match the feature table exactly:
1. the target is really *tomorrow's* AQI,
2. `AQI_lag1` is really *yesterday's* AQI,
3. `AQI_roll7` really covers the 7 days *before* today.

In [10]:
city = feat['City'].value_counts().index[0]
s = df[df['City'] == city].set_index('Date')['AQI'].asfreq('D')
chk = feat[feat['City'] == city].sample(min(200, (feat['City'] == city).sum()), random_state=0)
one, seven = pd.Timedelta(days=1), pd.Timedelta(days=7)

ok_target = all(np.isclose(s[d + one], t) for d, t in zip(chk['Date'], chk['target_AQI_next']))
ok_lag1 = all(np.isclose(s[d - one], v) for d, v in zip(chk['Date'], chk['AQI_lag1']))
ok_roll7 = all(np.isclose(s[d - seven:d - one].mean(), v) for d, v in zip(chk['Date'], chk['AQI_roll7']))

print(f"Checked {len(chk)} rows of {city}")
print("target = tomorrow's AQI  :", ok_target)
print("AQI_lag1 = yesterday's   :", ok_lag1)
print("AQI_roll7 excludes today :", ok_roll7)
assert ok_target and ok_lag1 and ok_roll7, "Leakage or alignment problem - do not continue"

assert not any('next' in c for c in ALL_FEATURES), "A feature name contains 'next'"
print("\nNo feature column refers to tomorrow. Features used:", len([c for c in ALL_FEATURES]))

Checked 200 rows of Agra
target = tomorrow's AQI  : True
AQI_lag1 = yesterday's   : True
AQI_roll7 excludes today : True

No feature column refers to tomorrow. Features used: 21


## Step 10 — Sanity checks and the baseline to beat
The simplest forecast is **persistence**: "tomorrow = today". Your models must beat it to be useful.


In [11]:
print("Target class counts (tomorrow's category):")
print(feat['target_bucket_next'].value_counts(), "\n")

print("Correlation today vs tomorrow:", round(feat['AQI'].corr(feat['target_AQI_next']), 3))
print("Persistence MAE (AQI points) :", round((feat['AQI'] - feat['target_AQI_next']).abs().mean(), 1))
print("Persistence accuracy (bucket):", round((feat['AQI_Bucket'] == feat['target_bucket_next']).mean(), 3))

Target class counts (tomorrow's category):
target_bucket_next
Satisfactory    64608
Moderate        58960
Good            28921
Poor            16246
Very Poor        4346
Severe            383
Name: count, dtype: int64 

Correlation today vs tomorrow: 0.88
Persistence MAE (AQI points) : 22.3
Persistence accuracy (bucket): 0.714


### Baseline Findings

Current-day and next-day AQI have a correlation of **0.88**, indicating substantial day-to-day persistence.

The persistence baseline, which predicts tomorrow's AQI using today's value, achieves:

- **Regression MAE:** 22.3 AQI units.
- **Classification accuracy:** 71.4%.

These results establish reference performance levels for evaluating whether more complex forecasting models provide additional predictive value.

## Step 11 — Save the feature table

In [12]:
os.makedirs("../data/Cleaned", exist_ok=True)
feat.to_csv(OUT_PATH, index=False)
print("Saved", OUT_PATH, "| shape:", feat.shape)
print("Date range:", feat['Date'].min().date(), "to", feat['Date'].max().date())

Saved ../data/Cleaned/aqi_2023_25_features.csv | shape: (173464, 25)
Date range: 2022-04-10 to 2025-04-29


## Summary and Key Findings

Feature engineering produced a modelling dataset containing **173,464 observations and 25 columns**, covering **10 April 2022 to 29 April 2025**.

The main outcomes are:

- Daily calendar alignment prevents missing dates from distorting lag definitions.
- Lagged AQI, rolling statistics, calendar indicators, and contextual variables capture different aspects of air-quality variation.
- Next-day regression and classification targets are generated separately from the predictors.
- Validation checks confirmed correct next-day target alignment, lag construction, and exclusion of current-day AQI from the tested rolling window.
- Strong temporal persistence establishes a meaningful baseline for forecasting.

The feature-engineered dataset is used in **Notebook 04 — Modelling**, where regression and classification algorithms are evaluated against these baselines.

### Limitations

Filtering incomplete observations reduces the available sample and may disproportionately exclude cities with irregular reporting. Furthermore, the dataset contains pollutant labels rather than measured pollutant concentrations, limiting the environmental detail available to the models.